# Spark Lab 入門

這個 notebook 連到 Spark 叢集（`spark://spark-master:7077`，2 個 worker）。
- 課程原始資料：`/data/course`（唯讀）
- 輸出 / 放大資料：`/work`
- 課程程式（重寫版）：`/opt/lab/jobs`，可以直接 `import`

執行時可以開 [Spark UI](/spark/) 看 job。

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName('jupyter playground')
         .config('spark.driver.host', 'jupyter')
         .config('spark.cores.max', '8')      # 留一半核心給控制台
         .config('spark.executor.memory', '2g')
         .getOrCreate())
sc = spark.sparkContext
spark

## RDD：WordCount（0715 第一支程式）

In [ ]:
import re
lines = sc.textFile('/data/course/spark101/wordcount/data/war_and_peace.txt')
words = lines.flatMap(lambda line: re.compile(r'\W+', re.UNICODE).split(line.lower())).filter(bool)
word_counts = words.map(lambda w: (w, 1)).reduceByKey(lambda x, y: x + y)
word_counts.sortBy(lambda p: -p[1]).take(10)

## DataFrame / Spark SQL：倫敦犯罪資料

In [ ]:
crime = spark.read.csv('/data/course/spark_sql_101/crime/data/london_crime_by_lsoa.csv', header=True, inferSchema=True)
crime.createOrReplaceTempView('crime')
spark.sql('SELECT borough, SUM(value) AS total FROM crime GROUP BY borough ORDER BY total DESC LIMIT 10').show()

## 期末專題的輸出表格（先在控制台跑完「期末專題 ①②③」）

In [ ]:
import os
T = '/work/ec_web_logs_analysis/tables'
for t in sorted(os.listdir(T)) if os.path.isdir(T) else []:
    print(t, spark.read.parquet(f'{T}/{t}').count())

In [ ]:
spark.stop()